# Activity 2 — SPO Triple Extraction

Build the core Information Extraction function that converts input sentences
into **Subject-Predicate-Object triples** with Wikidata Q/P IDs.

Key components:
- `PhraseMatcher` for gazetteer-based entity/relation detection
- `DependencyMatcher` for grammatical role assignment
- Word2Vec similarity for entity resolution
- Wikidata lookup for knowledge grounding

## 1. Setup

In [ ]:
# !pip install spacy numpy
# !python -m spacy download en_core_web_lg

import json, pathlib
import spacy
from spacy.matcher import PhraseMatcher, DependencyMatcher

DATA_DIR = pathlib.Path('../data')
nlp = spacy.load('en_core_web_lg')
print('spaCy pipeline:', nlp.pipe_names)

## 2. Load Gazetteers

In [ ]:
with open(DATA_DIR / 'entities_dict.json') as f:
    entities_dict = json.load(f)
with open(DATA_DIR / 'relations_dict.json') as f:
    relations_dict = json.load(f)

print(f"{len(entities_dict)} entities, {len(relations_dict)} relation phrases")

## 3. Build PhraseMatcher

In [ ]:
# TODO: paste your PhraseMatcher code from Colab here
# Reference implementation is in src/triple_extractor.py → build_phrase_matcher()

entity_matcher = PhraseMatcher(nlp.vocab, attr='LOWER')
relation_matcher = PhraseMatcher(nlp.vocab, attr='LOWER')

entity_patterns = [nlp.make_doc(e) for e in entities_dict.keys()]
relation_patterns = [nlp.make_doc(r) for r in relations_dict.keys()]

entity_matcher.add('ENTITY', entity_patterns)
relation_matcher.add('RELATION', relation_patterns)

## 4. SPO Extraction Function

In [ ]:
# TODO: paste your extract_triples_from_sentence() from Colab here
# The portable version lives in src/triple_extractor.py

import sys
sys.path.insert(0, '..')
from src.triple_extractor import extract_triples_from_sentence

## 5. Test on Sample Sentences

In [ ]:
test_sentences = [
    "Ripple partnered with Visa to improve their cross-border payment system.",
    "Ethereum was developed by Vitalik Buterin.",
    "IBM invested in Stellar to advance enterprise blockchain solutions.",
]

for sent in test_sentences:
    triples = extract_triples_from_sentence(sent, nlp=nlp,
                                            entities=entities_dict,
                                            relations=relations_dict)
    print(f"\nSentence: {sent}")
    for t in triples:
        print(f"  ({t['subject']}) --[{t['predicate']}]--> ({t['object']})")

## 6. Run on Full Corpus

In [ ]:
with open(DATA_DIR / 'sentences.txt') as f:
    corpus = [l.strip() for l in f if l.strip() and not l.startswith('#')]

all_triples = []
for sent in corpus:
    triples = extract_triples_from_sentence(sent, nlp=nlp,
                                            entities=entities_dict,
                                            relations=relations_dict)
    all_triples.extend(triples)

print(f"Extracted {len(all_triples)} triples from {len(corpus)} sentences")